In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv(r"C:\Users\zeyne\Desktop\veri_setleri\hourly_transportation_202410.csv")

df.head()

,transition_date,transition_hour,transport_type_id,road_type,line,transfer_type,number_of_passage,number_of_passenger,product_kind,transaction_type_desc,town,line_name,station_poi_desc_cd
0,2024-10-01,0,1,OTOYOL,CEBECI - TAKSIM,Normal,3,3,TAM,Tam Kontur,SARIYER,36T,NaN
1,2024-10-01,0,1,OTOYOL,USKUDAR-GUZELTEPE-UMRANIYE DEVLET HASTANESI,Aktarma,2,2,INDIRIMLI1,Indirimli Aktarma,BAKIRKOY,15B,NaN
2,2024-10-01,0,1,OTOYOL,SARIYER-HACIOSMAN-MECIDIYEKOY-TAKSIM,Normal,1,1,TAM,Tam Abonman,KAGITHANE,25G,NaN
3,2024-10-01,0,1,OTOYOL,YESILPINAR - ALIBEYKOY METRO,Aktarma,1,1,TAM,Tam Aktarma,BAKIRKOY,TM10,NaN
4,2024-10-01,0,1,OTOYOL,UMRANIYE DEV.HAST-CAKMAK MAH-ATASEHIR-USTBOSTANCI,Aktarma,1,1,INDIRIMLI2,Indirimli Tip 2 Aktarma,BAKIRKOY,10,NaN


Veri seti CSV dosyasından okunur.
Analiz ve modelleme için gerekli temel sütunlar seçilir.
Sütun adları daha anlaşılır ve standart bir formata dönüştürülür.
Tarih, saat, yolcu sayısı gibi alanların veri tipleri doğru tipe çevrilir.
Metin içeren sütunlar temizlenip (trim, upper/lower) tutarlı hale getirilir.
Bu işlemler sonucunda veri seti temiz, düzenli ve analiz için uygun duruma getirilir

In [2]:
df = pd.read_csv(r"C:\Users\zeyne\Desktop\veri_setleri\hourly_transportation_202410.csv")

df = df[['transition_date', 'transition_hour', 'line', 'road_type',
         'number_of_passenger', 'transport_type_id', 'town']]

df = df.rename(columns={
    'transition_date': 'date',
    'transition_hour': 'hour',
    'line': 'route_code',
    'road_type': 'stop_code',  # durak kodu yerine geçiyor
    'number_of_passenger': 'passenger_count',
    'transport_type_id': 'vehicle_type',
    'town': 'district'
})

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['hour'] = pd.to_numeric(df['hour'], errors='coerce').astype('Int64')

df['route_code'] = df['route_code'].astype(str).str.strip().str.upper()
df['stop_code'] = df['stop_code'].astype(str).str.strip().str.upper()
df['vehicle_type'] = df['vehicle_type'].astype(str).str.strip().str.lower()
df['passenger_count'] = pd.to_numeric(df['passenger_count'], errors='coerce')

print("Sütunlar dönüştürüldü ")
display(df.head(5))

Sütunlar dönüştürüldü 


,date,hour,route_code,stop_code,passenger_count,vehicle_type,district
0,2024-10-01,0,CEBECI - TAKSIM,OTOYOL,3,1,SARIYER
1,2024-10-01,0,USKUDAR-GUZELTEPE-UMRANIYE DEVLET HASTANESI,OTOYOL,2,1,BAKIRKOY
2,2024-10-01,0,SARIYER-HACIOSMAN-MECIDIYEKOY-TAKSIM,OTOYOL,1,1,KAGITHANE
3,2024-10-01,0,YESILPINAR - ALIBEYKOY METRO,OTOYOL,1,1,BAKIRKOY
4,2024-10-01,0,UMRANIYE DEV.HAST-CAKMAK MAH-ATASEHIR-USTBOSTANCI,OTOYOL,1,1,BAKIRKOY


DataFrame'deki toplam satır sayısını saklıyoruz.
"before" → temizleme işleminden önceki satır sayısı
Aynı tarih, saat, hat kodu, yol tipi ve araç türüne sahip tekrar eden satırlar kaldırılır.
Temizleme işleminden sonraki yeni satır sayısı.
Kaç tane yinelenen (duplicate) kaydın silindiğini ekrana yazdırıyoruz.

In [3]:
before = len(df)
df = df.drop_duplicates(subset=['date','hour','route_code','stop_code','vehicle_type'])
after = len(df)
print(f"Yinelenen kayıt sayısı: {before - after}")

Yinelenen kayıt sayısı: 2701358


DataFrame'deki tüm sütunlarda kaç adet eksik (NaN) değer olduğunu ekrana yazdırır.
Her bir route_code (hat) ve hour (saat) kombinasyonu için,
passenger_count içindeki eksik değerler o grubun "median" (ortanca) değeri ile doldurulur.
transform → aynı boyutta bir sonuç döndürerek gruplu doldurmayı sağlar.
Eğer hala eksik kalan varsa (örneğin medyan bulunamayan durumlarda),
tüm passenger_count sütunu için genel median ile dolduruyoruz.
passenger_count içerisinde eksik değer kalıp kalmadığını kontrol ediyoruz.

In [4]:
print(df.isna().sum())

df['passenger_count'] = df.groupby(['route_code','hour'])['passenger_count'].transform(
    lambda x: x.fillna(x.median())
)
df['passenger_count'] = df['passenger_count'].fillna(df['passenger_count'].median())

print("\nEksik değer kalmadı mı?:", df['passenger_count'].isna().sum() == 0)

date                 0
hour                 0
route_code           0
stop_code            0
passenger_count      0
vehicle_type         0
district           103
dtype: int64

Eksik değer kalmadı mı?: True


DataFrame içindeki tüm sütun adlarını bir liste halinde döndürür.

In [5]:
df.columns.tolist()

['date',
 'hour',
 'route_code',
 'stop_code',
 'passenger_count',
 'vehicle_type',
 'district']

Negatif yolcu sayılarını siler
Saat 0-23 aralığında olmayanları siler

In [6]:
df = df[(df['passenger_count'] >= 0)]

df = df[df['hour'].between(0, 23)]

'date' sütununu datetime formatına dönüştürüyoruz.
Veri setinindeki aylara göre filtreliyoruz.

In [7]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

df = df[(df['date'] >= "2024-10-01") & (df['date'] <= "2024-10-31")]

Kategorik veri tipine dönüştürülmesini istediğimiz sütunları liste halinde tanımlıyoruz.
Bu sütunlar: hat kodu, yol tipi, araç türü ve ilçe bilgisidir.
Listedeki her sütun için döngü başlatıyoruz
Her kolonu 'category' veri tipine dönüştürüyoruz.

In [8]:
cat_cols = ['route_code', 'stop_code', 'vehicle_type', 'district']

for col in cat_cols:
    df[col] = df[col].astype("category")

cat_cols içindeki kategorik sütunların tamamını ele alıyoruz ve apply ile hepsine işlem uyguluyoruz.

In [9]:
df[cat_cols] = df[cat_cols].apply(lambda x: x.str.strip() if x.dtype == "object" else x)

Tüm sütunları göz önünde bulundurarak veri setindeki tekrar eden satırları siler.
Temizleme işleminden sonra DataFrame'in yeni boyutunu (satır, sütun) ekrana yazdırır.

In [10]:
df = df.drop_duplicates()
print("Yeni boyut:", df.shape)

Yeni boyut: (69361, 7)


SMOTE Yöntemi Veri Kaybı Minimumu
Mevcut Dağılımı gösterir.

In [11]:
print("\n ÖNCEKİ DAĞILIM:")
original_counts = df['vehicle_type'].value_counts()
print(original_counts)
print("\n Yüzdelik Dağılım:")
print(df['vehicle_type'].value_counts(normalize=True) * 100)


 ÖNCEKİ DAĞILIM:
vehicle_type
1    63861
3     3230
2     2270
Name: count, dtype: int64

 Yüzdelik Dağılım:
vehicle_type
1    92.070472
3     4.656796
2     3.272733
Name: proportion, dtype: float64


En büyük sınıfın örnek sayısı bulunur.Çoğunluk sınıfını %50 azaltarak hedef boyut belirlenir.
Her sınıf için hedef kayıt sayısı gösterilir.

In [12]:
max_class_size = original_counts.max()
majority_reduced = int(max_class_size * 0.5)

print(f"1️ En büyük sınıf boyutu: {max_class_size:,} (Veri kaybı: 0)")
print(f"2️ %50 azaltılmış boyut: {majority_reduced:,} (Tavsiye edilen)")
print(f"3️ Ortalama: {int(original_counts.mean()):,}")

target_size = majority_reduced

print(f"\n SEÇİLEN HEDEF: {target_size:,} kayıt/sınıf")


1️ En büyük sınıf boyutu: 63,861 (Veri kaybı: 0)
2️ %50 azaltılmış boyut: 31,930 (Tavsiye edilen)
3️ Ortalama: 23,120

 SEÇİLEN HEDEF: 31,930 kayıt/sınıf


Orijinal veriyi bozmamak için kopya oluşturuldu ve her kategorik kolon için Label Encoding uygulandı.
Sayısal feature'lar belirlendi ve encode edilmiş kategorik feature'lar listelendi.
Ek feature'lar eklendi. Kullanılacak feature listesini yazdırıldı.
Girdi (X) ve hedef (y) değişkenleri ayırıldı ve SMOTE için X'in float64 tipinde olması gerekir.
Veri boyutlarını ve tiplerini kontrol edilir.

In [13]:
categorical_cols = ['route_code', 'stop_code', 'district']
label_encoders = {}

df_encoded = df.copy()

for col in categorical_cols:
    if col in df_encoded.columns:
        le = LabelEncoder()
        df_encoded[col + '_encoded'] = le.fit_transform(df_encoded[col].astype(str))
        label_encoders[col] = le

numeric_features = ['hour', 'passenger_count']
encoded_features = [col + '_encoded' for col in categorical_cols if col in df_encoded.columns]

extra_features = []
if 'day_of_week' in df_encoded.columns:
    extra_features.append('day_of_week')
if 'is_weekend' in df_encoded.columns:
    extra_features.append('is_weekend')
if 'is_peak_hour' in df_encoded.columns:
    extra_features.append('is_peak_hour')

feature_columns = numeric_features + encoded_features + extra_features

print(f"Kullanılacak feature'lar: {feature_columns}")

X = df_encoded[feature_columns].astype('float64')
y = df_encoded['vehicle_type']

print(f" X shape: {X.shape}")
print(f" y shape: {y.shape}")
print(f" X dtype'ları: {X.dtypes.unique()}")

Kullanılacak feature'lar: ['hour', 'passenger_count', 'route_code_encoded', 'stop_code_encoded', 'district_encoded']
 X shape: (69361, 5)
 y shape: (69361,)
 X dtype'ları: [dtype('float64')]


Her sınıf için örnekleri tutmak için liste oluşturuldu.İlgili taşıma türü sınıfa ait verileri seçildi. 
Çoğunluk sınıfını hedef boyuta kadar rastgele azaltıldı. Azınlık sınıfını SMOTE ile artırılacak.
İşlenen sınıf listeye eklendi. Tüm sınıflar birleştirildi. 
Feature matrisi SMOTE için float64 olmalı. Her sınıfın hedef boyutu tanımlandı. SMOTE nesnesini oluşturuldu.
SMOTE uygulayarak dengeli veri üretildi.

In [14]:
df_resampled_list = []

for vehicle_type in df_encoded['vehicle_type'].unique():
    df_class = df_encoded[df_encoded['vehicle_type'] == vehicle_type]
    current_size = len(df_class)
    
    if current_size > target_size:
        df_class_sampled = df_class.sample(n=target_size, random_state=42, replace=False)
        print(f" {vehicle_type}: {current_size:,} → {target_size:,} (undersampling)")
    else:
        df_class_sampled = df_class
        print(f" {vehicle_type}: {current_size:,} → {current_size:,} (bekliyor)")
    
    df_resampled_list.append(df_class_sampled)

df_undersampled = pd.concat(df_resampled_list, ignore_index=True)

print(f"\n Undersampling sonrası toplam: {len(df_undersampled):,}")

X_under = df_undersampled[feature_columns].astype('float64')  
y_under = df_undersampled['vehicle_type']

sampling_strategy = {cls: target_size for cls in y_under.unique()}

smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=5)
X_balanced, y_balanced = smote.fit_resample(X_under, y_under)

print(f" SMOTE tamamlandı!")

 1: 63,861 → 31,930 (undersampling)
 2: 2,270 → 2,270 (bekliyor)
 3: 3,230 → 3,230 (bekliyor)

 Undersampling sonrası toplam: 37,430
 SMOTE tamamlandı!


SMOTE çıktısı olan X_balanced'ı DataFrame formatına çevrildi. Sayısal kolonları uygun tipe dönüştürüldü.
Sayısal kolonları uygun tipe dönüştürüldü.Encode edilmiş kolonları orijinal kategorik değerlere çevirildi.
Sayısal kodları orijinal kategorik etiketlere dönüştürüldü ve gerek olmayan encoded kolonu silindi.

In [15]:
df_balanced = pd.DataFrame(X_balanced, columns=feature_columns)

integer_cols = ['hour', 'passenger_count'] + encoded_features + extra_features
for col in integer_cols:
    if col in df_balanced.columns:
        df_balanced[col] = df_balanced[col].round().astype('int64')

df_balanced['vehicle_type'] = y_balanced

for col in categorical_cols:
    if col in df_encoded.columns:
        encoded_col = col + '_encoded'
        if encoded_col in df_balanced.columns:
        
            df_balanced[col] = label_encoders[col].inverse_transform(
                df_balanced[encoded_col].astype(int)
            )
            df_balanced = df_balanced.drop(encoded_col, axis=1)

print(f"\n Final DataFrame oluşturuldu!")


 Final DataFrame oluşturuldu!


Orijinal ve dengelenmiş veri seti boyutlarını yazdırıldı. Veri kaybı / artışı miktarı hesaplandı.
Veri kaybı mı yoksa artışı mı olduğunu kontrol edildi ve dengeleme sonrası sınıf dağılımını gösterildi.

In [16]:
print(f"\n ÖNCEKİ TOPLAM: {len(df):,}")
print(f" SONRAKİ TOPLAM: {len(df_balanced):,}")

veri_kaybi = len(df) - len(df_balanced)
veri_kaybi_yuzdesi = (veri_kaybi / len(df)) * 100

if veri_kaybi > 0:
    print(f"\n Veri Kaybı: {veri_kaybi:,} ({veri_kaybi_yuzdesi:.2f}%)")
else:
    print(f"\n Veri Artışı: {abs(veri_kaybi):,} ({abs(veri_kaybi_yuzdesi):.2f}%)")

print(f"\n SONRAKI DAĞILIM:")
final_counts = df_balanced['vehicle_type'].value_counts()
print(final_counts)

print("\n Yüzdelik Dağılım:")
print(df_balanced['vehicle_type'].value_counts(normalize=True) * 100)


 ÖNCEKİ TOPLAM: 69,361
 SONRAKİ TOPLAM: 95,790

 Veri Artışı: 26,429 (38.10%)

 SONRAKI DAĞILIM:
vehicle_type
1    31930
2    31930
3    31930
Name: count, dtype: int64

 Yüzdelik Dağılım:
vehicle_type
1    33.333333
2    33.333333
3    33.333333
Name: proportion, dtype: float64


 Yolcu sayısı sütununun alt çeyrek (Q1) değerini hesaplar.
 Yolcu sayısı sütununun üst çeyrek (Q3) değerini hesaplar.
 IQR (Interquartile Range) yani çeyrekler arası genişlik.
 passenger_count değeri alt/üst sınırların dışındaysa AYKIRIDIR.
 Kaç tane aykırı (1) ve normal (0) değer olduğunu sayar.

In [17]:
Q1 = df_balanced['passenger_count'].quantile(0.25)
Q3 = df_balanced['passenger_count'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df_balanced['is_outlier'] = ((df_balanced['passenger_count'] < lower) | (df_balanced['passenger_count'] > upper)).astype(int)


'hour' sütunundaki her bir değere bir fonksiyon (lambda) uyguluyoruz.
 Saat 07:00–10:00 arası sabah yoğunluğu
 Saat 16:00–20:00 arası akşam yoğunluğu
 Bu aralıklardaysa 1, değilse 0 döndürür.

In [18]:
df_balanced['is_peak_hour'] = df_balanced['hour'].apply(lambda x: 1 if (7 <= x <= 10) or (16 <= x <= 20) else 0)

route_code (hat kodu) ve hour (saat) bilgisine göre gruplama yapıyoruz.
passenger_count sütununun grup içindeki ortalamasını hesaplıyoruz.


In [19]:
df_balanced['mean_passenger_by_route_hour'] = (
    df_balanced.groupby(['route_code', 'hour'], observed=True)['passenger_count']
    .transform('mean')
)

DataFrame'i CSV formatında belirtilen dosya adıyla dışa aktarıyoruz.
index=False → satır numaralarının (index) dosyaya yazılmasını engeller.
Dosyanın başarıyla kaydedildiğini kullanıcıya bildiriyoruz.

In [20]:
df_balanced.to_csv("istanbul_trans_balanced_clean_10.csv", index=False)
print("Kaydedildi ")


Kaydedildi 
